# === STEP 1: Mount Google Drive ===

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# === STEP 2: Install Required Packages ===

In [ ]:
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install torch-geometric

# === STEP 3: Import Libraries ===

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

# === STEP 4: Define File Paths ===

In [ ]:
data_root = "path"
train_dir = os.path.join(data_root, "path")
test_dir = os.path.join(data_root, "path")
metadata_path = os.path.join(data_root, "path")
test_metadata_path = os.path.join(data_root, "path")

# === STEP 5: Define Data Extraction Functions ===

In [ ]:
def extract_upper_triangle(file_path):
    matrix = pd.read_csv(file_path, sep='\t', header=None).values
    upper_tri_indices = np.triu_indices_from(matrix, k=1)
    return matrix[upper_tri_indices]

def extract_full_matrix(file_path):
    matrix = pd.read_csv(file_path, sep='\t', header=None).values
    return matrix[np.newaxis, :, :]

# === STEP 6: Load and Merge Training Data ===

In [ ]:
train_vectors, train_ids = [], []
print("🔄 Processing training files...")
for filename in tqdm(os.listdir(train_dir)):
    if filename.endswith(".tsv"):
        participant_id = filename.split("_")[0].replace("sub-", "").replace(".tsv", "")
        file_path = os.path.join(train_dir, filename)
        vector = extract_upper_triangle(file_path)
        train_vectors.append(vector)
        train_ids.append(participant_id)

X_train_raw = pd.DataFrame(train_vectors)
X_train_raw["participant_id"] = train_ids
metadata = pd.read_csv(metadata_path)
X_train_raw["participant_id"] = X_train_raw["participant_id"].str.upper().str.strip()
metadata["participant_id"] = metadata["participant_id"].str.upper().str.strip()
train_df = pd.merge(X_train_raw, metadata, on="participant_id")

# === STEP 7: Oversample to Balance Age Distribution ===

In [ ]:
# Group data by age ranges and oversample minority age groups more heavily
older = train_df[train_df.age >= 17]
middle = train_df[(train_df.age >= 13) & (train_df.age < 17)]
younger = train_df[train_df.age < 13]

# Oversample minority groups
older_oversampled = resample(older, replace=True, n_samples=300, random_state=42)
middle_oversampled = resample(middle, replace=True, n_samples=400, random_state=42)

# Combine into final resampled DataFrame
df_resampled = pd.concat([younger, middle_oversampled, older_oversampled]).reset_index(drop=True)

# Recalculate age bins after resampling
age_bins = pd.cut(df_resampled["age"], bins=[7, 11, 14, 17, 21], labels=False, include_lowest=True)

age_bins = pd.cut(df_resampled["age"], bins=[7, 11, 14, 17, 21], labels=False, include_lowest=True)

# Remove invalid rows before any transformations
good_mask = ~age_bins.isna()
df_resampled = df_resampled[good_mask].reset_index(drop=True)
age_bins = age_bins[good_mask].reset_index(drop=True)

# === STEP 8: Feature Engineering ===

In [ ]:
brain_cols = [col for col in df_resampled.columns if isinstance(col, int) or (isinstance(col, str) and col.isdigit())]
meta_cols = [col for col in df_resampled.columns if col not in brain_cols + ["participant_id", "age"]]

scaler = StandardScaler()
brain_scaled = scaler.fit_transform(df_resampled[brain_cols])
pca = PCA(n_components=100)
brain_pca = pca.fit_transform(brain_scaled)

meta_encoded = pd.get_dummies(df_resampled[meta_cols])
meta_encoded = meta_encoded.reindex(columns=meta_encoded.columns, fill_value=0)

X = np.concatenate([brain_pca, meta_encoded.values], axis=1)
y = df_resampled["age"]

# === STEP 9: Target Transformation ===

In [ ]:
qt = QuantileTransformer(output_distribution='normal')

# Clean y and age_bins
age_bins = pd.cut(df_resampled["age"], bins=[7, 11, 14, 17, 21], labels=False, include_lowest=True)
non_nan_mask = (~df_resampled["age"].isna()) & (~age_bins.isna())
df_resampled = df_resampled.loc[non_nan_mask].reset_index(drop=True)
age_bins = age_bins.loc[non_nan_mask].reset_index(drop=True)

# Recompute features to align with cleaned data
brain_scaled = scaler.fit_transform(df_resampled[brain_cols])
brain_pca = pca.fit_transform(brain_scaled)
meta_encoded = pd.get_dummies(df_resampled[meta_cols])
meta_encoded = meta_encoded.reindex(columns=meta_encoded.columns, fill_value=0)
X = np.concatenate([brain_pca, meta_encoded.values], axis=1)

# Transform target
y = df_resampled["age"]
y_transformed = qt.fit_transform(y.values.reshape(-1, 1)).ravel()

# Final mask for any leftover NaNs from transformation
good_idx = ~np.isnan(y_transformed)
X = X[good_idx]
y_transformed = y_transformed[good_idx]
y = y.reset_index(drop=True)[good_idx]
age_bins = age_bins[good_idx].reset_index(drop=True)

# === STEP 10: Train/Val/Test Split ===

In [ ]:
# First split data (X and y) with stratification based on age bins
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_transformed, test_size=0.30, stratify=age_bins, random_state=42
)

# Now split the age_bins array to use for further stratification
bins_train, bins_temp = train_test_split(
    age_bins, test_size=0.30, stratify=age_bins, random_state=42
)
bins_val, bins_test = train_test_split(
    bins_temp, test_size=0.50, stratify=bins_temp, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=bins_temp, random_state=42
)

# === STEP 11: Impute Missing Values ===

In [ ]:
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)
X_test = imputer.transform(X_test)

# === STEP 12: Train XGBoost ===

In [ ]:
# Recalculate weights: give more importance to rarer ages
age_freq = df_resampled["age"].value_counts(normalize=True)
df_resampled["weight"] = df_resampled["age"].map(lambda x: 1 / age_freq[x])
sample_weights = df_resampled.iloc[:len(X_train)]["weight"].values

xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

# === STEP 13: Evaluation ===

In [ ]:
def inverse(y): return qt.inverse_transform(y.reshape(-1, 1)).ravel()
def eval_metrics(y_true, y_pred, name):
    print(f"\n📊 {name} Set")
    print("R²:", r2_score(y_true, y_pred))
    print("MSE:", mean_squared_error(y_true, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))
    print("MAE:", mean_absolute_error(y_true, y_pred))

for split, X_, y_ in zip(["Train", "Val", "Test"], [X_train, X_val, X_test], [y_train, y_val, y_test]):
    preds = inverse(xgb_model.predict(X_))
    y_actual = inverse(y_)
    eval_metrics(y_actual, preds, split)

# === STEP 14: Process Test Data ===

In [ ]:
test_vectors, test_ids = [], []
for filename in tqdm(os.listdir(test_dir)):
    if filename.endswith(".tsv"):
        participant_id = filename.split("_")[0].replace("sub-", "").replace(".tsv", "")
        vector = extract_upper_triangle(os.path.join(test_dir, filename))
        test_vectors.append(vector)
        test_ids.append(participant_id)

X_test_raw = pd.DataFrame(test_vectors)
X_test_raw["participant_id"] = test_ids
test_metadata = pd.read_csv(test_metadata_path)
X_test_raw["participant_id"] = X_test_raw["participant_id"].str.upper().str.strip()
test_metadata["participant_id"] = test_metadata["participant_id"].str.upper().str.strip()
test_df = pd.merge(X_test_raw, test_metadata, on="participant_id")

# === STEP 15: Prepare Final Submission ===

In [ ]:
test_scaled = scaler.transform(test_df[brain_cols])
test_pca = pca.transform(test_scaled)
test_meta_encoded = pd.get_dummies(test_df[meta_cols])
test_meta_encoded = test_meta_encoded.reindex(columns=meta_encoded.columns, fill_value=0)
X_submit = np.concatenate([test_pca, test_meta_encoded.values], axis=1)

raw_preds = xgb_model.predict(X_submit)

# === STEP 15b: Stretch Predictions to Compensate for Low Age Bias ===

In [ ]:
from scipy.stats import rankdata

def stretch_predictions(preds, min_age=8, max_age=21):
    quantiles = rankdata(preds) / len(preds)
    return min_age + quantiles * (max_age - min_age)

preds = stretch_predictions(raw_preds)
preds = np.clip(preds, 8, 21)

submission = pd.DataFrame({"participant_id": test_df["participant_id"], "age": preds})
submission_path = os.path.join(data_root, "new_submission.csv")
submission.to_csv(submission_path, index=False)
print(f"✅ Submission saved to {submission_path}")

# === STEP 16: Plot Prediction Distribution ===

In [ ]:
sns.histplot(preds, bins=20, kde=True)
plt.title("Predicted Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load metadata
metadata_path = "path"
train_metadata = pd.read_csv(metadata_path)

# === 1. Age Distribution Table ===
age_stats = train_metadata["age"].value_counts().sort_index().reset_index()
age_stats.columns = ["age", "count"]
age_stats["percentage"] = 100 * age_stats["count"] / age_stats["count"].sum()

# Display the age statistics table
print("\n📊 Age Distribution Table:")
print(age_stats)

# === 2. Summary Statistics ===
print("\n📈 Summary Stats:")
print(train_metadata["age"].describe())

# === 3. Optional: Plot Age Distribution ===
plt.figure(figsize=(10, 5))
sns.histplot(train_metadata["age"], bins=14, kde=True)
plt.title("Age Distribution in Training Set")
plt.xlabel("Age")
plt.ylabel("Count")
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd

# Load the training metadata CSV file
metadata_path = "path"
df = pd.read_csv(metadata_path)

# Count the number of samples per age
age_distribution = df["age"].value_counts().sort_index().reset_index()
age_distribution.columns = ["age", "count"]

# Optionally add percentage column
age_distribution["percentage"] = 100 * age_distribution["count"] / age_distribution["count"].sum()

# Display the results
print(age_distribution)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
sns.barplot(data=age_distribution, x="age", y="count", palette="viridis")
plt.title("Sample Count per Age")
plt.xlabel("Age")
plt.ylabel("Number of Samples")
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd

# Load the training metadata
metadata_path = "path"
df = pd.read_csv(metadata_path)

# Convert float ages to integer by flooring (or use round if preferred)
df["age_int"] = df["age"].astype(int)

# Count the number of samples per integer age
age_distribution = df["age_int"].value_counts().sort_index().reset_index()
age_distribution.columns = ["age", "count"]

# Add percentage column
age_distribution["percentage"] = 100 * age_distribution["count"] / age_distribution["count"].sum()

# Display the results
print(age_distribution)